# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [2]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [3]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

🏃 View run rare-robin-264 at: http://localhost:5000/#/experiments/530798079091162401/runs/c68ade20e609404287b6a5a3be32b5c9
🧪 View experiment at: http://localhost:5000/#/experiments/530798079091162401


Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [4]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2025/07/22 14:02:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/22 14:02:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'sk-learn-random-forest-reg-model' already exists. Creating a new version of this model...
2025/07/22 14:02:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 3


🏃 View run selective-stork-509 at: http://localhost:5000/#/experiments/530798079091162401/runs/b68c744765bb4803b62beb7687650815
🧪 View experiment at: http://localhost:5000/#/experiments/530798079091162401


Created version '3' of model 'sk-learn-random-forest-reg-model'.


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [5]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

/home/iraitz/TheBridge/DSPT2025-ML/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.49980732  44.5541425   41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.44675489 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [6]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [7]:
#!pip install hyperopt

In [8]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

2025-07-22 14:02:18.427380: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-22 14:02:18.439958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753185738.452621 3783981 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753185738.456024 3783981 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753185738.466204 3783981 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [9]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [10]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

2025/07/22 14:02:20 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/805050304746925943', creation_time=1753185740204, experiment_id='805050304746925943', last_update_time=1753185740204, lifecycle_stage='active', name='wine-quality', tags={}>

In [12]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 21s 488ms/step - loss: 27.5947 - root_mean_squared_error: 5.2531
28/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.7430 - root_mean_squared_error: 5.2671   
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 27.7941 - root_mean_squared_error: 5.2720
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 27.7967 - root_mean_squared_error: 5.2722 - val_loss: 27.7259 - val_root_mean_squared_error: 5.2655

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 28.7238 - root_mean_squared_error: 5.3595
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 27.6121 - root_mean_squared_error: 5.2546 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.5874 - root_mean_squared_error: 5.2523 - val_loss: 26.8501 - val_root_mean_squared_error: 5.1817

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 26.3838 - root_mean_squared_error: 5.

2025/07/22 14:03:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:03:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run respected-steed-392 at: http://localhost:5000/#/experiments/805050304746925943/runs/cb3f749c2de549f7a4793b85dc3dad17

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 23s 530ms/step - loss: 37.0077 - root_mean_squared_error: 6.0834
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 34.9698 - root_mean_squared_error: 5.9123   
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 34.1190 - root_mean_squared_error: 5.8393
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 34.0718 - root_mean_squared_error: 5.8353 - val_loss: 28.3019 - val_root_mean_squared_error: 5.3200

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 26.3870 - root_mean_squared_error: 5.1368
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 26.9286 - root_mean_squared_error: 5.1890 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3

2025/07/22 14:03:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:03:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run grandiose-kite-697 at: http://localhost:5000/#/experiments/805050304746925943/runs/ff687eb8a30b42a19c9008a2b5081d6c

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943 

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 24s 540ms/step - loss: 31.4387 - root_mean_squared_error: 5.6070
30/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.6290 - root_mean_squared_error: 2.4199    
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 5.1635 - root_mean_squared_error: 2.1184
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 5.0993 - root_mean_squared_error: 2.1045 - val_loss: 0.8049 - val_root_mean_squared_error: 0.8972

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.6183 - root_mean_squared_error: 0.7863
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6781 - root_mean_squared_error: 0.8234 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/st

...

2025/07/22 14:04:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:04:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run awesome-conch-115 at: http://localhost:5000/#/experiments/805050304746925943/runs/b24e3301f0c745ada3a4d0958e6caee8

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943  

100%|██████████| 8/8 [00:51<00:00,  6.40s/trial, best loss: 0.7440080642700195]

2025/07/22 14:04:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2025/07/22 14:04:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best parameters: {'lr': np.float64(0.05252363303162457), 'momentum': np.float64(0.12216694910176285)}
Best eval rmse: 0.7440080642700195
🏃 View run unequaled-sponge-797 at: http://localhost:5000/#/experiments/805050304746925943/runs/6a676b408eac49e3b6fb4febdefdbb4e
🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [13]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 22s 490ms/step - loss: 37.8906 - root_mean_squared_error: 6.1555
39/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.3354 - root_mean_squared_error: 2.3327    
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 5.7301 - root_mean_squared_error: 2.2113
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 5.6561 - root_mean_squared_error: 2.1962 - val_loss: 0.7401 - val_root_mean_squared_error: 0.8603

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.8871 - root_mean_squared_error: 0.9419
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6696 - root_mean_squared_error: 0.8179 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6628 - root_mean_squared_error: 0.8138 - val_loss: 0.5723 - val_root_mean_squared_error: 0.7565

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.4077 - root_mean_squared_error: 0.6385
46/

2025/07/22 14:05:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:05:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run puzzled-horse-845 at: http://localhost:5000/#/experiments/805050304746925943/runs/4441f42038ea4c0fba86b0d84f1291d1

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 24s 549ms/step - loss: 38.6644 - root_mean_squared_error: 6.2181
38/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 24.7435 - root_mean_squared_error: 4.9165   
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 22.7472 - root_mean_squared_error: 4.6947
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 22.5270 - root_mean_squared_error: 4.6697 - val_loss: 2.3348 - val_root_mean_squared_error: 1.5280

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.1601 - root_mean_squared_error: 1.4697
37/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.1440 - root_mean_squared_error: 1.4633 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/

2025/07/22 14:05:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:05:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run burly-fish-478 at: http://localhost:5000/#/experiments/805050304746925943/runs/e893fcba53fb423daf254d696aaa3efa

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943  

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 23s 531ms/step - loss: 36.8490 - root_mean_squared_error: 6.0703
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 37.4348 - root_mean_squared_error: 6.1181   
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 36.8675 - root_mean_squared_error: 6.0712
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 36.8260 - root_mean_squared_error: 6.0677 - val_loss: 31.3094 - val_root_mean_squared_error: 5.5955

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 33.0023 - root_mean_squared_error: 5.7448
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 29.3979 - root_mean_squared_error: 5.4212 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3m

...

2025/07/22 14:05:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2025/07/22 14:05:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



🏃 View run whimsical-rook-670 at: http://localhost:5000/#/experiments/805050304746925943/runs/ce9efe2939ec4a3baf6d1c9d811e14d6

🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943  

100%|██████████| 8/8 [00:52<00:00,  6.51s/trial, best loss: 0.7596985697746277]

2025/07/22 14:05:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2025/07/22 14:06:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best parameters: {'lr': np.float64(0.05847417595381574), 'momentum': np.float64(0.10999444312704498)}
Best eval rmse: 0.7596985697746277
🏃 View run mercurial-hen-323 at: http://localhost:5000/#/experiments/805050304746925943/runs/61b9a1f58b904665b0cbd545abc2a4b5
🧪 View experiment at: http://localhost:5000/#/experiments/805050304746925943


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [21]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

200


In [22]:
response.content

b'{"predictions": [[6.128170013427734]]}'